In [32]:
import pandas as pd

In [33]:
df = pd.read_csv("nsdi26ae.csv")

## Finding 1: The majority (52.2%) of the studied operator failures are caused by defects in operators’ interactions with external entities which significantly outnumbers bugs in operators’ internal program logic.

In [34]:
interaction_count = df[df["Root Cause Location"] == "Interaction"].shape[0]
total_count = df.shape[0]
percentage = (interaction_count / total_count * 100) if total_count > 0 else 0
print(f"The majority ({percentage:.1f}%) of the studied operator failures are "
      "caused by defects in operators' interactions with external entities "
      "which significantly outnumbers bugs in operators' internal program logic.")

The majority (52.2%) of the studied operator failures are caused by defects in operators' interactions with external entities which significantly outnumbers bugs in operators' internal program logic.


## Finding 2: The majority (62.3%) of interaction-related operator failures had catastrophic impacts, e.g., application full outages, partial outages, and data loss.

In [35]:
full_outages = df[df["Consequence"] == "Application Outage"].shape[0]
partial_outages = df[df["Consequence"] == "Application Partial Outage"].shape[0]
data_loss = df[df["Consequence"] == "Application Data Loss"].shape[0]

percentage = (
    (full_outages + partial_outages + data_loss) / interaction_count * 100
    if interaction_count > 0 else 0
)
print(f"The majority ({percentage:.1f}%) of interaction-related operator failures "
      "had catastrophic impacts, e.g., application full outages, partial outages, "
      "and data loss.")

The majority (62.3%) of interaction-related operator failures had catastrophic impacts, e.g., application full outages, partial outages, and data loss.


## Figure 3: Root-cause locations of the studied 412 operator failures (3a), and the consequences of the 215 interaction-related operator failures (3b).

In [ ]:
# Figure 3a: Root-cause locations of the studied 412 operator failures with counts and percentages
print("\Figure 3a: Root-cause locations of the studied 412 operator failures with counts and percentages")
root_cause_counts = df["Root Cause Location"].value_counts()
root_cause_percentages = (root_cause_counts / total_count * 100).round(1)
root_cause_table = pd.DataFrame({
    "Count": root_cause_counts,
    "Percentage": root_cause_percentages
}).reset_index().rename(columns={"index": "Root Cause Location"})

root_cause_table

\Figure 3a: Root-cause locations of the studied 412 operator failures with counts and percentages


<>:2: SyntaxWarning: invalid escape sequence '\F'
<>:2: SyntaxWarning: invalid escape sequence '\F'
/var/folders/s6/qw6z7w7j59z68ccqfmx53yc00000gp/T/ipykernel_67841/1397748490.py:2: SyntaxWarning: invalid escape sequence '\F'
  print("\Figure 3a: Root-cause locations of the studied 412 operator failures with counts and percentages")


,Root Cause Location,Count,Percentage
0,Interaction,215,52.2
1,Internal,104,25.2
2,HelmChart,57,13.8
3,Misuse,36,8.7


In [37]:
# Figure 3b

interaction_failures = df[df["Root Cause Location"] == "Interaction"]

# group by consequence with count and percentage in the same table
consequence_counts = interaction_failures["Consequence"].value_counts()
consequence_percentages = (
    consequence_counts / interaction_count * 100 if interaction_count > 0 else 0
)
consequence_summary = pd.DataFrame({
    "Count": consequence_counts,
    "Percentage": consequence_percentages.round(1)
}).reset_index().rename(columns={"index": "Consequence"})

print("Figure 3b: the consequences of the 215 interaction-related operator failures")
consequence_summary

Figure 3b: the consequences of the 215 interaction-related operator failures


,Consequence,Count,Percentage
0,Application Outage,108,50.2
1,Application Misconfiguration,29,13.5
2,Application Partial Outage,19,8.8
3,Operator Outage,18,8.4
4,Reliability Issue,16,7.4
5,Performance Issue,13,6.0
6,Application Data Loss,7,3.3
7,Security Issue,5,2.3


## Finding 3: Failures of managing applications is the largest category (42.3%) among all operator interaction failures.

In [38]:
interaction_failures = df[df["Root Cause Location"] == "Interaction"]
application_count = interaction_failures[interaction_failures["Interaction"] == "Application"]
application_percentage = (application_count.shape[0] / interaction_count * 100) if interaction_count > 0 else 0
print(f"Failures of managing applications is the largest category ({application_percentage:.1f}%) among all operator interaction failures.")

Failures of managing applications is the largest category (42.3%) among all operator interaction failures.


## Table 3: Distribution of different types of interaction failures of the studied operators.

In [39]:
interaction_failures = df[df["Root Cause Location"] == "Interaction"]
interaction_failures_by_operator = interaction_failures.groupby("Operator")["Interaction"].value_counts().unstack(fill_value=0)
interaction_failures_by_operator["Total"] = interaction_failures_by_operator.sum(axis=1)
interaction_failures_by_operator.loc["Total"] = interaction_failures_by_operator.sum()
interaction_failures_by_operator = interaction_failures_by_operator[["Application", "Co-operator", "Platform", "User", "Total"]]
print("Table 3: Distribution of different types of interaction failures of the studied operators.")
interaction_failures_by_operator

Table 3: Distribution of different types of interaction failures of the studied operators.


Interaction,Application,Co-operator,Platform,User,Total
Operator,,,,,
CN/PostgresOp,15,0,1,4,20
CassOp,2,1,0,5,8
CockroachOp,4,0,6,2,12
KafkaOp,7,0,9,5,21
KnativeOp,0,1,4,8,13
KubeBlocks,12,0,7,1,20
MinIOOp,3,1,5,7,16
MongoOp,15,0,4,1,20
RabbitMQOp,3,1,10,0,14


## Finding 4: We find four main failure patterns:
- The majority (63.7%) of studied application-management failures are caused by the operator violating its managed application’s operation semantics.
- A significant percentage (16.5%) of management failures are caused by the gaps that prevent the operator from observing application internal states.
- Incompatibility between the operator and its managed application also caused a significant percentage (12.1%) of failures, triggered by upgrading application versions.
- The remaining cases (7.7%) were caused by the operator mishandling application errors.

In [36]:
# The majority (63.7%) of studied application-management failures are caused by the operator violating its managed application’s operation semantics.
semantic_violations = application_count[application_count["Pattern"] == "Semantic violations"]
semantic_violations_percentage = (semantic_violations.shape[0] / application_count.shape[0] * 100) if application_count.shape[0] > 0 else 0
print(f"The majority ({semantic_violations_percentage:.1f}%) of studied application-management failures are caused by the operator violating its managed application's operation semantics.")

# A significant percentage (16.5%) of management failures are caused by the gaps that prevent the operator from observing application internal states.
observation_gaps = application_count[application_count["Pattern"] == "State observability"]
observation_gaps_percentage = (observation_gaps.shape[0] / application_count.shape[0] * 100) if application_count.shape[0] > 0 else 0
print(f"A significant percentage ({observation_gaps_percentage:.1f}%) of management failures are caused by the gaps that prevent the operator from observing application internal states.")

# Incompatibility between the operator and its managed application also caused a significant percentage (12.1%) of failures, triggered by upgrading application versions.
incompatibility = application_count[application_count["Pattern"] == "Version incompatibility"]
incompatibility_percentage = (incompatibility.shape[0] / application_count.shape[0] * 100) if application_count.shape[0] > 0 else 0
print(f"Incompatibility between the operator and its managed application also caused a significant percentage ({incompatibility_percentage:.1f}%) of failures, triggered by upgrading application versions.")

# The remaining cases (7.7%) were caused by the operator mishandling application errors.
error_mishandling = application_count[application_count["Pattern"] == "Error handling"]
error_mishandling_percentage = (error_mishandling.shape[0] / application_count.shape[0] * 100) if application_count.shape[0] > 0 else 0
print(f"The remaining cases ({error_mishandling_percentage:.1f}%) were caused by the operator mishandling application errors.")

The majority (63.7%) of studied application-management failures are caused by the operator violating its managed application's operation semantics.
A significant percentage (16.5%) of management failures are caused by the gaps that prevent the operator from observing application internal states.
Incompatibility between the operator and its managed application also caused a significant percentage (12.1%) of failures, triggered by upgrading application versions.
The remaining cases (0.0%) were caused by the operator mishandling application errors.


## Table 4: Patterns of operator-application interaction failures

In [40]:
# group by pattern and count occurrences
pattern_counts = application_count["Pattern"].value_counts()
pattern_percentages = (
    pattern_counts / application_count.shape[0] * 100 if application_count.shape[0] > 0 else 0
)
pattern_summary = pd.DataFrame({
    "Count": pattern_counts,
    "Percentage": pattern_percentages.round(1)
}).reset_index().rename(columns={"index": "Pattern"})
print("Table 4: Patterns of operator-application interaction failures")
pattern_summary

Table 4: Patterns of operator-application interaction failures


,Pattern,Count,Percentage
0,Semantic violations,58,63.7
1,State observability,15,16.5
2,Version incompatibility,11,12.1
3,Mishandling application errors,7,7.7


## Table 5: The types of violated operation semantics.

In [42]:
operation_semantics = application_count[application_count["Pattern"] == "Semantic violations"]
operation_semantics_types = operation_semantics["Operation Semantics"].value_counts()
# display the total 
total_semantics = operation_semantics_types.sum()
operation_semantics_types = operation_semantics_types.reset_index()
operation_semantics_types.columns = ["Operation Semantics", "Count"]
operation_semantics_types["Percentage"] = (operation_semantics_types["Count"] / total_semantics * 100).round(1)
operation_semantics_types = operation_semantics_types.sort_values(by="Count", ascending=False).reset_index(drop=True)
print("Table 5: The types of violated operation semantics.")
print(operation_semantics_types)

Table 5: The types of violated operation semantics.
  Operation Semantics  Count  Percentage
0       Configuration     21        36.2
1            Ordering     18        31.0
2        Precondition     11        19.0
3         Environment      8        13.8
